In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
import warnings

# Dual-Sensor UAV Anomaly Detection: Exploratory Data Analysis

## 1. Environment Initialization and Data Ingestion
To begin our analysis, we must first establish a computational environment and ingest our raw telemetry and network datasets. We rely on standard data science libraries, including Pandas and NumPy, for matrix operations and data manipulation. For our visual diagnostic profiling, we configure Seaborn and Matplotlib to ensure that all generated plots are automatically saved to our `plots/` directory. Furthermore, we import statistical modules from SciPy and Statsmodels, which will later allow us to perform hypothesis testing, such as the Augmented Dickey-Fuller test for time-series stationarity. 

Once the environment is stabilized, we load our DVC-tracked datasets: the **SurveilDrone-Net23** (Kinematic dataset) representing the physical flight states, and the **drone_communication_dataset** (Tokyo Network dataset) representing the cyber-communication logs. The immediate objective is to inspect the raw dimensional boundaries of our matrices, denoted as $X_{kinematic}$ and $X_{cyber}$, alongside their initial feature naming conventions. This initial inspection dictates our subsequent data cleaning and intersection strategies.

In [ ]:
# Visual settings
sns.set_theme(style = "whitegrid", context = "paper", font_scale = 1.2)
plt.rcParams['figure.figsize'] = (12, 6)

print("Environment successfully initialized.\n")

# Load datasets (adjust paths if your DVC structure differs)
df_kin = pd.read_csv('../data/SurveilDrone-Net23.csv')
df_net = pd.read_csv('../data/drone_communication_dataset.csv')

# Inspect matrix dimensions
print(f"Kinematic Dataset Shape (X_kinematic): {df_kin.shape}")
print(f"Network Dataset Shape (X_cyber): {df_net.shape}\n")

# Inspect raw column names
print(f"Raw Kinematic Columns: {df_kin.columns.tolist()[:8]}...")
print(f"Raw Network Columns: {df_net.columns.tolist()[:8]}...")

The initial ingestion reveals a dimensional asymmetry between our two data matrices, which informs our architectural strategy. The Kinematic matrix ($X_{kinematic}$) contains **140,256 instances** across **34 spatial and physical features**, whereas the Cyber matrix ($X_{cyber}$) is substantially smaller, containing **52,585 instances** across **29 network features**. This discrepancy suggests a difference in sampling frequencies; physical telemetry (like velocity and altitude) is likely logged at a higher hertz rate than network communication packets, which are recorded upon transmission events. 

Furthermore, inspection of the feature vectors reveals **inconsistent naming conventions**. The Kinematic dataset utilizes a standard `snake_case` format (e.g., `altitude_m`, `velocity_x`), while the Network dataset relies on capitalized strings with spaces (e.g., `Packet Size`, `Signal Strength`). Crucially, both datasets contain a temporal identifier (`timestamp` and `Timestamp`) alongside drone routing identifiers. To build a **Dual-Sensor Pipeline**, we must first normalize these inconsistencies to prevent key-errors and evaluate the presence of missing values ($NaN$), which would inherently break the distance calculations required for our **Isolation Forest** and **Local Outlier Factor** algorithms.

## 2. Feature Standardization and Data Integrity Profiling
To achieve a stable environment, we must apply a transformation to our feature space. We define a vectorised cleaning function that strips trailing whitespace, converts all characters to lowercase, removes special alphanumeric characters, and replaces spaces with underscores. This ensures that every column across both $X_{kinematic}$ and $X_{cyber}$ adheres to a **Pythonic standard**. 

Following this normalization, we conduct a global null-value evaluation. **Unsupervised anomaly detection** algorithms, particularly those relying on **Euclidean** or **Mahalanobis distances**, are strictly incompatible with incomplete feature vectors. Identifying the sparsity of our matrices at this stage determines whether we must employ imputation techniques, such as **K-Nearest Neighbors** imputation or **forward-filling** ($x_t = x_{t-1}$), or if the datasets are structurally complete.

In [ ]:
def standardize_columns(df):
    """
    Applies vectorised string transformations to normalize feature names.
    Converts to lowercase, removes special characters, and replaces spaces.
    """
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(r'[^a-zA-Z0-9\s_]', '', regex = True)
        .str.replace(r'\s+', '_', regex = True)
    )
    return df

# Apply transformations to both matrices
df_kin = standardize_columns(df_kin)
df_net = standardize_columns(df_net)

# Output the standardized feature spaces
print("Standardized Kinematic Features:", df_kin.columns.tolist()[:8], "...")
print("Standardized Network Features:", df_net.columns.tolist()[:8], "...\n")

# Calculate global matrix sparsity (null values)
kin_nulls = df_kin.isnull().sum().sum()
net_nulls = df_net.isnull().sum().sum()

print(f"Global Sparsity (Missing Values) in X_kinematic: {kin_nulls}")
print(f"Global Sparsity (Missing Values) in X_cyber: {net_nulls}")

# Check for duplicate rows which could artificially dense clusters
print(f"Duplicate vectors in X_kinematic: {df_kin.duplicated().sum()}")
print(f"Duplicate vectors in X_cyber: {df_net.duplicated().sum()}")

The execution of our standardization confirms that the feature spaces for both matrices have been successfully mapped to a uniform **Pythonic schema**, aligning the temporal identifiers as `timestamp` across both datasets. Crucially, the global sparsity evaluation yields exactly **zero missing values** ($NaN$) and **zero duplicate vectors**. From a mathematical perspective, this structural completeness is **highly advantageous**. **Unsupervised distance-based** algorithms, such as the **Local Outlier Factor (LOF)**, rely heavily on neighborhood density estimations. Duplicate rows artificially inflate local density, masking true inliers, while missing values break **Euclidean and Mahalanobis distance** calculations. Because our matrices are dense and unique, we can bypass synthetic imputation layers and proceed directly to temporal fusion.

## 3. Temporal Alignment and Sampling Frequency Analysis
Because we are making a **Dual-Sensor Pipeline**, the **physical telemetry** and **cyber-communication logs** must be synchronized into a single, unified chronological index. However, before executing a matrix join, we must verify their temporal boundaries and sampling frequencies ($\Delta t$). 

**Our objective here is:**
1. Parse the string-based `timestamp` arrays into standard Pandas datetime objects.
2. Extract the temporal boundaries ($T_{start}$ and $T_{end}$) for both matrices to quantify their exact overlap duration.
3. Compute the median time delta between consecutive logs to determine the sampling frequency (e.g., $10Hz$ versus $1Hz$). If the frequencies exhibit high variance, we will need to aggregate or interpolate the data using our previously discussed rolling window strategy (Hypothesis 3).

In [ ]:
# Convert timestamps to datetime objects
df_kin['timestamp'] = pd.to_datetime(df_kin['timestamp'])
df_net['timestamp'] = pd.to_datetime(df_net['timestamp'])

# Sort both matrices chronologically to ensure temporal continuity
df_kin = df_kin.sort_values(by='timestamp').reset_index(drop=True)
df_net = df_net.sort_values(by='timestamp').reset_index(drop=True)

# 1. Calculate Temporal Boundaries
kin_start, kin_end = df_kin['timestamp'].min(), df_kin['timestamp'].max()
net_start, net_end = df_net['timestamp'].min(), df_net['timestamp'].max()

kin_duration = kin_end - kin_start
net_duration = net_end - net_start

print("--- Temporal Boundaries ---")
print(f"Kinematic Timeline: {kin_start} to {kin_end} (Duration: {kin_duration})")
print(f"Network Timeline:   {net_start} to {net_end} (Duration: {net_duration})\n")

# 2. Calculate Overlap
latest_start = max(kin_start, net_start)
earliest_end = min(kin_end, net_end)

if latest_start <= earliest_end:
    overlap = earliest_end - latest_start
    print(f"Intersection Achieved: Datasets overlap for {overlap}.")
else:
    print("Warning: Datasets do not share a chronological intersection.")

# 3. Calculate Sampling Frequencies (Median Time Delta)
kin_delta = df_kin['timestamp'].diff().median()
net_delta = df_net['timestamp'].diff().median()

print("\n--- Sampling Frequencies ---")
print(f"Kinematic Median Delta: {kin_delta}")
print(f"Network Median Delta:   {net_delta}")

The temporal boundary evaluation reveals a critical structural asymmetry between our two data sources. While we have successfully isolated a viable intersection window of **1,095 days** (spanning from January 1, 2021, to January 1, 2024), there is a significant discrepancy in their sampling frequencies. The Kinematic telemetry logs states **every 15 minutes** ($f_{kin} = 1/900$ Hz), whereas the Network matrix logs communication health **every 60 minutes** ($f_{net} = 1/3600$ Hz). 

This 4:1 frequency ratio presents a profound architectural challenge. A naive temporal inner-join would result in an immediate 75% data loss for the kinematic dataset, or conversely, introduce 75% sparsity ($NaN$ values) into the network columns if an outer-join is used. Because unsupervised models rely on spatial density, introducing synthetic sparsity is flawed. Instead, we will apply a downsampling technique to the Kinematic matrix. By aggregating the 15-minute physical telemetry into 1-hour temporal buckets using the arithmetic mean $\mu = \frac{1}{n} \sum_{i=1}^{n} x_i$, we inherently satisfy our "Temporal State Hypothesis" (Hypothesis 3). This aggregation acts as a low-pass filter, smoothing out instantaneous physical micro-glitches and isolating the sustained structural behaviors required to detect anomalies.

## 4. Matrix Truncation and Multi-Sensor Fusion
Having established the theoretical basis for our alignment, we now construct the unified state matrix ($X_{dual}$). The operation sequence is strictly defined:
1. Truncate both $X_{kinematic}$ and $X_{cyber}$ to the rigid **1,095-day** intersection boundary.
2. Align the relational keys by standardising the drone identification variables (mapping `source_drone_id` to `drone_id`).
3. Execute the **1-hour temporal downsampling** on the kinematic space, aggregating continuous physical variables to match the network logging frequency.
4. Perform an inner-join on the multi-index key (`timestamp`, `drone_id`) to produce the final, mathematically dense multi-variate feature space.

In [ ]:
# 1. Isolate the intersection boundaries
start_bound = max(kin_start, net_start)
end_bound = min(kin_end, net_end)

df_kin_sync = df_kin[(df_kin['timestamp'] >= start_bound) & (df_kin['timestamp'] <= end_bound)].copy()
df_net_sync = df_net[(df_net['timestamp'] >= start_bound) & (df_net['timestamp'] <= end_bound)].copy()

# Rename network ID column to match
if 'source_drone_id' in df_net_sync.columns:
    df_net_sync.rename(columns = {'source_drone_id': 'drone_id'}, inplace=True)

# 2. Relational Key Normalization: Force IDs to strings and strip non-numeric characters
df_kin_sync['drone_id'] = df_kin_sync['drone_id'].astype(str).str.replace(r'\D', '', regex=True)
df_net_sync['drone_id'] = df_net_sync['drone_id'].astype(str).str.replace(r'\D', '', regex=True)

# 3. Temporal Alignment: Round Network timestamps to the nearest hour
df_net_sync['timestamp'] = df_net_sync['timestamp'].dt.round('h')

# 4. Temporal Downsampling (15m -> 1H) for continuous Kinematic features
numeric_kin_cols = df_kin_sync.select_dtypes(include = [np.number]).columns
numeric_kin_cols = [col for col in numeric_kin_cols if col not in ['timestamp', 'drone_id']]

# Group and resample kinematic data
df_kin_resampled = (
    df_kin_sync.set_index('timestamp')
    .groupby('drone_id')[numeric_kin_cols]
    .resample('h')
    .mean()
    .reset_index()
)

# 5. Execute the Multi-Sensor Fusion
df_fused = pd.merge(
    df_kin_resampled, 
    df_net_sync, 
    on = ['timestamp', 'drone_id'], 
    how = 'inner'
)

print("--- Fusion Diagnostics ---")
print(f"Kinematic IDs (Sample): {df_kin_sync['drone_id'].unique()[:3]}")
print(f"Network IDs (Sample):   {df_net_sync['drone_id'].unique()[:3]}\n")

print("--- Successful Fusion ---")
print(f"Unified Matrix Shape (X_dual): {df_fused.shape}")
print(f"Remaining Missing Values: {df_fused.isnull().sum().sum()}")

# Save a snapshot of the fused raw matrix for our DVC tracking pipeline
df_fused.to_csv('../data/fused_telemetry_network.csv', index = False)

## 5. Matrix Sparsity Purging and Final Density
The **multi-sensor fusion** yielded a unified matrix of **26,278 operational events** across **54 combined kinematic** and **network features**. However, the fusion introduced a substantial degree of sparsity **(429,275 $NaN$ values)**. This sparsity is an inherent mathematical byproduct of the temporal `.resample('1H')` operation; forcing asynchronous physical telemetry into continuous 1-hour temporal buckets generates "empty" intervals for periods where a drone was grounded or powered offline, but its networking hardware continued to register standby communication logs.

**Unsupervised distance-based** algorithms construct hyperplanes and measure cluster densities using complete feature vectors. Injecting synthetic data (e.g., using mean imputation or forward-filling) to guess a drone's physical velocity or altitude during these offline periods would alter the standard deviation of our baseline state, directly corrupting the mathematical boundaries of our **Isolation Forest**. Consequently, our strategy is **Listwise Deletion**. We will purge any temporal bucket containing incomplete feature vectors, collapsing $X_{dual}$ into a dense, active state matrix. Once purged, we will visually profile the multi-variate distributions of the fused dataset to ensure stability before moving to feature engineering.

In [ ]:
# 1. Listwise Deletion: Purge all incomplete temporal buckets
df_fused_clean = df_fused.dropna().reset_index(drop=True)

# 2. Verify Final Matrix Density
final_shape = df_fused_clean.shape
final_nulls = df_fused_clean.isnull().sum().sum()

print("--- Final Matrix Density Verification ---")
print(f"Cleaned Matrix Shape: {final_shape}")
print(f"Remaining Missing Values: {final_nulls}\n")

# 3. Multi-Variate Distribution Profiling (Validating the Dual-Sensor Pipeline)
# We will randomly select one kinematic feature and one cyber feature to ensure 
# the distributions remain mathematically sound after the fusion and purging.

# Select numeric features dynamically to avoid hardcoding errors
numeric_cols = df_fused_clean.select_dtypes(include=[np.number]).columns
kin_sample_feat = [col for col in numeric_cols if col in df_kin_resampled.columns][0]
net_sample_feat = [col for col in numeric_cols if col in df_net_sync.columns and col != 'timestamp'][0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Kinematic Distribution
sns.histplot(df_fused_clean[kin_sample_feat], ax=axes[0], color='blue', bins=50)
axes[0].set_title(f"Kinematic State Density: {kin_sample_feat}")
axes[0].set_ylabel("Frequency")

# Plot Network Distribution
sns.histplot(df_fused_clean[net_sample_feat], ax=axes[1], color='red', bins=50)
axes[1].set_title(f"Cyber State Density: {net_sample_feat}")
axes[1].set_ylabel("Frequency")

plt.suptitle("Dual-Sensor Fusion: Post-Purge Operational Distributions", y=1.05, fontsize=14)
plt.tight_layout()
plt.savefig('../plots/fused_state_distributions.png')
plt.show()

# Update the DVC-tracked final clean matrix
df_fused_clean.to_csv('../data/fused_clean_pipeline.csv', index = False)

The execution of our sparsity purging protocol has successfully condensed $X_{dual}$ into a strictly dense, continuous matrix of 9,107 joint operational events. From an algorithmic perspective, this sample size is highly optimal, providing sufficient density for neighborhood-based algorithms to map local hyperplanes without introducing excessive computational complexity. 

Crucially, the generated density plots reveal a stark asymmetry between the physical and cyber feature spaces. The kinematic state (represented by `altitude_m`) exhibits a continuous, relatively platykurtic (flat) probability distribution, indicating smooth, analog transitions across the drone's operational flight ceiling. Conversely, the cyber state (represented by `packet_size`) demonstrates a highly discrete, multimodal distribution. Network traffic does not scale continuously; it is constrained by byte-size protocols (e.g., 500, 1000, 1500 MTU), resulting in massive density spikes. 

This visual disparity validates our choice of unsupervised machine learning over simple thresholding. A traditional algorithm might handle the continuous physical data well but would struggle with the discrete network spikes. Tree-based models like the Isolation Forest, however, partition feature spaces using orthogonal splits, making them immune to these discrete structural variances, perfectly setting the stage for our dual-sensor hypothesis testing.

## 6. Univariate Boundary Hunting and Contamination Baselining (Hypothesis 1 & 2 Setup)
Before engineering complex multi-variate machine learning models, we must first establish a baseline to prove *why* those models are necessary. This aligns directly with **Experiment 1 (Raw Baseline)** and our project's goal of isolating the top 5% contamination rate. 

If attacks and physical failures could be easily detected by a single metric going out of bounds, we would not need Machine Learning; we could just write a simple `if/else` threshold rule. To prove **Hypothesis 1 (Kinematic)** and **Hypothesis 2 (Cyber)**, we will perform **Univariate Boundary Hunting**. We will calculate the 95th percentile ($P_{95}$) and the 5th percentile ($P_{05}$) for critical features to isolate the statistical 5% tails. We will then plot these static boundaries to demonstrate how single-metric alerts fail to capture complex, multi-variable anomalies (e.g., a drone that is operating within normal altitude bounds, but draining battery too fast for that specific altitude).

In [ ]:
# Define the target contamination rate as dictated by the project parameters
contamination_rate = 0.05
lower_bound_pct = contamination_rate / 2
upper_bound_pct = 1 - (contamination_rate / 2)

# Select one critical Kinematic and one Cyber feature for baselining
kin_target = 'altitude_m' if 'altitude_m' in df_fused_clean.columns else numeric_cols[0]
net_target = 'packet_size' if 'packet_size' in df_fused_clean.columns else numeric_cols[-1]

def plot_univariate_boundaries(df, feature, lower_pct, upper_pct, color):
    """Calculates and plots the statistical 5% boundaries for a given feature."""
    lower_val = df[feature].quantile(lower_pct)
    upper_val = df[feature].quantile(upper_pct)
    
    plt.figure(figsize=(10, 4))
    sns.histplot(df[feature], bins=50, kde=False, color=color, alpha=0.6)
    
    # Plot operational boundaries
    plt.axvline(lower_val, color = 'red', linestyle = '--', linewidth = 2, label = f'Lower 2.5% ({lower_val:.2f})')
    plt.axvline(upper_val, color = 'red', linestyle = '--', linewidth = 2, label = f'Upper 2.5% ({upper_val:.2f})')
    
    # Highlight the anomalous regions
    plt.axvspan(df[feature].min(), lower_val, color='red', alpha=0.2)
    plt.axvspan(upper_val, df[feature].max(), color='red', alpha=0.2)
    
    plt.title(f"Univariate Boundary Hunting: {feature} (5% Contamination Rate)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'../plots/univariate_baseline_{feature}.png')
    plt.show()
    
    return lower_val, upper_val

# Execute and plot boundaries
kin_lower, kin_upper = plot_univariate_boundaries(df_fused_clean, kin_target, lower_bound_pct, upper_bound_pct, 'blue')
net_lower, net_upper = plot_univariate_boundaries(df_fused_clean, net_target, lower_bound_pct, upper_bound_pct, 'gray')

print("--- Statistical Baseline Boundaries (Top 5% Anomalies) ---")
print(f"Kinematic ({kin_target}): Normal operations bounded between {kin_lower:.2f} and {kin_upper:.2f}")
print(f"Cyber ({net_target}): Normal operations bounded between {net_lower:.2f} and {net_upper:.2f}")

## 7. Multicollinearity Profiling and Feature Space Redundancy
Because we intend to deploy distance-based anomaly detection algorithms (such as the Local Outlier Factor and K-Means Hybrid), we must evaluate the matrix for multicollinearity. These algorithms rely heavily on Euclidean or Mahalanobis distance equations to calculate local cluster densities:
$$d(p,q) = \sqrt{\sum_{i=1}^{n} (p_i - q_i)^2}$$

If two features are perfectly correlated (e.g., $r = 1.0$), they mathematically represent the exact same vector of information. Including both in a distance calculation artificially inflates the spatial weight of that underlying trait by a factor of $\sqrt{2}$, warping the anomaly boundaries and leading to false positives. 

To preserve the geometric integrity of our models, we will compute the Pearson correlation coefficient matrix:
$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$
We will visualize this via a lower-triangle heatmap and programmatically isolate any feature pairs that exhibit a correlation threshold of $|r| > 0.85$. These redundant features must be flagged for removal during our feature engineering pipeline.

In [ ]:
# Isolate purely numeric features for mathematical correlation
numeric_fused = df_fused_clean.select_dtypes(include = [np.number])
corr_matrix = numeric_fused.corr()

# Generate a geometric mask for the upper triangle to remove visual redundancy
mask = np.triu(np.ones_like(corr_matrix, dtype = bool))

# Set up the matplotlib figure
plt.figure(figsize = (16, 14))
sns.heatmap(corr_matrix, mask = mask, cmap = 'coolwarm', vmax = 1, vmin = -1, 
            center = 0, square = True, linewidths = .5, cbar_kws = {"shrink": .5})

plt.title("Dual-Sensor Feature Space: Pearson Correlation Matrix", fontsize = 16)
plt.tight_layout()
plt.savefig('../plots/fused_correlation_matrix.png')
plt.show()

# iIsolate highly correlated feature pairs (|r| > 0.85)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > 0.85:
            colname_i = corr_matrix.columns[i]
            colname_j = corr_matrix.columns[j]
            correlation_val = corr_matrix.iloc[i, j]
            high_corr_pairs.append((colname_i, colname_j, correlation_val))

print(f"Total quantitative features evaluated: {len(numeric_fused.columns)}")
print("--- Highly Correlated Feature Pairs (|r| > 0.85) ---")

if not high_corr_pairs:
    print("No mathematically redundant feature pairs found.")
else:
    for pair in high_corr_pairs:
        print(f"Redundancy Flag: {pair[0]} & {pair[1]} (r = {pair[2]:.3f})")

The execution of the **Pearson correlation matrix** reveals a highly orthogonal feature space. Across all 47 quantitative variables evaluated, there are exactly zero pairs exhibiting a correlation coefficient exceeding the absolute threshold of $|r| > 0.85$. In the context of multi-sensor IoT telemetry, this is an exceptionally clean result. It implies that every engineered feature—from the physical kinematics to the cyber-communication logs—contributes distinct variance to the overall state matrix $X_{dual}$. 

Because there is no severe multicollinearity, we are completely protected against the geometric distortion of distance metrics. If, for instance, `velocity_x` and `acceleration_x` had yielded a correlation of $r = 0.98$, any distance-based algorithm (like K-Means or Isolation Forest) would implicitly double the weight of the forward-momentum vector, severely warping the hyperplanes that define our "normal" operational boundaries. Thanks to this inherent orthogonality, we do not need to execute arbitrary feature dropping or dimensionality reduction at this stage, preserving the maximum amount of contextual data for our anomaly detectors.